In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

df = pd.read_csv("training_comparison.csv")
cl = df[df["run"] == "CL-4"]
ib = df[df["run"] == "IB-4"]

In [ ]:
import os
os.makedirs("plots", exist_ok=True)

metrics = [
    ("R1@0.5", "Recall@1 IoU≥0.5 (↑)"),
    ("R1@0.7", "Recall@1 IoU≥0.7 (↑)"),
    ("mAP",    "Mean Average Precision (↑)"),
    ("mIoU",   "Mean IoU (↑)"),
    ("loss_f", "Foreground Loss (↓)"),
    ("loss_g", "Global Loss (↓)"),
]

paper_refs = {
    "R1@0.5": [("MATR Trained+V",   52.7, "#4CAF50", "--"),
               ("MATR Finetuned+V", 56.5, "#4CAF50", "-.")],
    "mIoU":   [("MATR Trained+V",   56.2, "#4CAF50", "--"),
               ("MATR Finetuned+V", 59.2, "#4CAF50", "-.")],
}

def make_plot(ax, col, title):
    ax.plot(cl["epoch"], cl[col], "o-", color="#2196F3", label="CLIP (CL-4)", linewidth=2, markersize=6)
    ax.plot(ib["epoch"], ib[col], "s-", color="#FF5722", label="ImageBind (IB-4)", linewidth=2, markersize=6)
    if col in paper_refs:
        for label, val, color, ls in paper_refs[col]:
            ax.axhline(val, color=color, linestyle=ls, linewidth=1.5, alpha=0.8, label=f"Paper: {label} ({val})")
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("Epoch")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    for data, color in [(cl, "#2196F3"), (ib, "#FF5722")]:
        best_idx = data[col].idxmin() if "loss" in col else data[col].idxmax()
        bx, by = data.loc[best_idx, "epoch"], data.loc[best_idx, col]
        ax.axvline(bx, color=color, linestyle="--", alpha=0.3)
        ax.annotate(f"{by:.2f}", xy=(bx, by), xytext=(4, 4),
                    textcoords="offset points", color=color, fontsize=9, fontweight="bold")

# Combined 2x3 grid
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle("CLIP vs ImageBind Training Comparison", fontsize=15, fontweight="bold")
for ax, (col, title) in zip(axes.flat, metrics):
    make_plot(ax, col, title)
plt.tight_layout()
plt.savefig("training_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved training_comparison.png")

# Individual plots
for col, title in metrics:
    fig, ax = plt.subplots(figsize=(7, 5))
    make_plot(ax, col, title)
    fig.suptitle("CLIP vs ImageBind — " + title, fontsize=12, fontweight="bold")
    plt.tight_layout()
    fname = f"plots/{col.replace('@', '_at_').replace('/', '_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved {fname}")

In [ ]:
print("=== Peak Metrics Summary ===")
for run_name, data in [("CLIP (CL-4)", cl), ("ImageBind (IB-4)", ib)]:
    best_epoch = data.loc[data["mAP"].idxmax(), "epoch"]
    print(f"\n{run_name} — best at epoch {best_epoch}:")
    for col in ["R1@0.5", "R1@0.7", "mAP", "mIoU"]:
        print(f"  {col:8s}: {data[col].max():.2f}")